# 04 - KAN vs MLP Comparison

This notebook directly compares KAN and MLP on the same tasks.
pykan includes a built-in `MLP` class with the same `.fit()` interface,
ensuring a fair comparison (same optimizer, loss, batching).

We compare:
- Loss curves
- Parameter counts
- Training time
- Generalization

In [ ]:
import time
import torch
from kan import KAN, MLP, create_dataset
import matplotlib.pyplot as plt

In [ ]:
f = lambda x: torch.sin(torch.pi * x[:, [0]]) + torch.cos(torch.pi * x[:, [1]])
dataset = create_dataset(f, n_var=2, train_num=1000, test_num=1000, ranges=[-1, 1])

## 1. Train KAN

In [ ]:
kan_model = KAN(width=[2, 5, 1], grid=5, k=3, seed=42, auto_save=False)
kan_params = sum(p.numel() for p in kan_model.parameters())

t0 = time.time()
kan_results = kan_model.fit(dataset, opt="LBFGS", steps=100, lr=1.0, lamb=0.001)
kan_time = time.time() - t0

print(f"KAN: {kan_params} params, {kan_time:.1f}s, final test RMSE: {kan_results['test_loss'][-1]:.6f}")

## 2. Train MLP with comparable capacity

In [ ]:
mlp_model = MLP(width=[2, 50, 1], seed=42)
mlp_params = sum(p.numel() for p in mlp_model.parameters())

t0 = time.time()
mlp_results = mlp_model.fit(dataset, opt="LBFGS", steps=100, lr=1.0, lamb=0.001)
mlp_time = time.time() - t0

print(f"MLP: {mlp_params} params, {mlp_time:.1f}s, final test RMSE: {mlp_results['test_loss'][-1]:.6f}")

## 3. Compare loss curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Training loss
axes[0].plot(kan_results["train_loss"], label=f"KAN ({kan_params} params)")
axes[0].plot(mlp_results["train_loss"], label=f"MLP ({mlp_params} params)")
axes[0].set_xlabel("Step")
axes[0].set_ylabel("Train RMSE")
axes[0].set_yscale("log")
axes[0].legend()
axes[0].set_title("Training Loss")

# Test loss
axes[1].plot(kan_results["test_loss"], label=f"KAN ({kan_params} params)")
axes[1].plot(mlp_results["test_loss"], label=f"MLP ({mlp_params} params)")
axes[1].set_xlabel("Step")
axes[1].set_ylabel("Test RMSE")
axes[1].set_yscale("log")
axes[1].legend()
axes[1].set_title("Test Loss")

plt.suptitle("KAN vs MLP on sin(πx) + cos(πy)")
plt.tight_layout()
plt.show()

## 4. Summary table

In [ ]:
import pandas as pd

summary = pd.DataFrame({
    "Model": ["KAN [2,5,1]", "MLP [2,50,1]"],
    "Parameters": [kan_params, mlp_params],
    "Final Train RMSE": [float(kan_results["train_loss"][-1]), float(mlp_results["train_loss"][-1])],
    "Final Test RMSE": [float(kan_results["test_loss"][-1]), float(mlp_results["test_loss"][-1])],
    "Training Time (s)": [kan_time, mlp_time],
})
summary

## 5. Multiple functions comparison

Compare KAN vs MLP across several target functions.

In [ ]:
functions = {
    "sin(πx) + cos(πy)": lambda x: torch.sin(torch.pi * x[:, [0]]) + torch.cos(torch.pi * x[:, [1]]),
    "x² + xy + y²": lambda x: x[:, [0]]**2 + x[:, [0]]*x[:, [1]] + x[:, [1]]**2,
    "exp(sin(πx) + y²)": lambda x: torch.exp(torch.sin(torch.pi * x[:, [0]]) + x[:, [1]]**2),
}

rows = []
for name, fn in functions.items():
    ds = create_dataset(fn, n_var=2, train_num=1000, test_num=1000)

    # KAN
    k = KAN(width=[2, 5, 1], grid=5, k=3, seed=42, auto_save=False)
    kr = k.fit(ds, opt="LBFGS", steps=100, lr=1.0, lamb=0.001)

    # MLP
    m = MLP(width=[2, 50, 1], seed=42)
    mr = m.fit(ds, opt="LBFGS", steps=100, lr=1.0, lamb=0.001)

    rows.append({
        "Function": name,
        "KAN Test RMSE": float(kr["test_loss"][-1]),
        "MLP Test RMSE": float(mr["test_loss"][-1]),
    })

comparison = pd.DataFrame(rows)
comparison